# Visual Table Assistant — Dataset Preparation

## Purpose

This notebook builds the final YOLO dataset for Phase 1 from the raw zips
stored on Google Drive. Output is a flat layout under
`datasets/table_assistant_yolo/` plus a manifest, a stratified train/val/test
split, and supporting reports.

Training does not happen here. Once this notebook finishes successfully and
the dataset has been pushed via DVC, training runs in
`02_training_colab.ipynb`.

## Detection classes (frozen)

The seven YOLO classes used end-to-end by the project are defined in
`configs/classes.yaml`. They are reproduced here for quick reference:

| ID | Class |
|---:|---|
| 0 | food |
| 1 | cup |
| 2 | bottle |
| 3 | plate |
| 4 | spoon |
| 5 | fork |
| 6 | knife |

`food` is generic. The system does not aim to identify the type of food.

## 1. Environment Setup

Verify Python and clone (or update) the project repository under `/content/`.
Cells use absolute paths because Colab sessions can disconnect mid-run, and a
`%cd` from a previous cell does not survive the reconnect; absolute paths
make every cell safe to re-run independently.

### 1.1 Checking Python Version

In [ ]:
import sys
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())

### 1.2 Project Path Variables

In [ ]:
REPO_URL = "https://github.com/LucasGVallejos/iaa-visual-table-assistant"
WORKSPACE_DIR = "/content"
PROJECT_DIR = "iaa-visual-table-assistant"
PROJECT_PATH = f"{WORKSPACE_DIR}/{PROJECT_DIR}"

### 1.3 Cloning GitHub Repository

In [ ]:
import os

%cd {WORKSPACE_DIR}

if not os.path.exists(PROJECT_PATH):
    !git clone {REPO_URL} {PROJECT_PATH}
else:
    print("Repository already exists. Pulling latest changes...")
    !git -C {PROJECT_PATH} pull

%cd {PROJECT_PATH}

### 1.4 Installing Dependencies

In [ ]:
!pip install -r requirements.txt

### 1.5 Sanity Check on Installed Libraries

In [ ]:
import torch
import ultralytics
import cv2
import mlflow
import onnx
import onnxruntime
import dvc
import numpy as np
import pandas as pd

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

print("OpenCV:", cv2.__version__)
print("ONNX Runtime:", onnxruntime.__version__)

## 2. Raw Dataset Setup from Google Drive

Mount Google Drive, extract the raw zips into the repo, inspect their structure,
and run a visual bounding-box sanity check before any conversion. All logic
lives in versioned scripts under `src/data/raw_setup/`.

**Expected Drive layout:**
```
MyDrive/iaa-table-assistant/
  raw_datasets/
    open_images/          # zip(s) from download_open_images_subset.py
    uec_food_256/         # zip(s) from UEC FOOD-256
```

### 2.1 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### 2.2 Extract and Inspect Raw Datasets

Locates the raw zips on Drive, extracts them into
`datasets/raw_datasets/` inside the repo, and prints a structural summary so
you can spot missing folders or unexpected layouts before continuing.

In [ ]:
!python -m src.data.raw_setup.setup_colab_raw_datasets

### 2.3 Visual Bounding Box Sanity Check (Raw)

Renders one sample image per source with its raw bounding boxes drawn on top.
Confirms that COCO and UEC bbox parsing is correct **before** any conversion
to YOLO format.

Outputs: `outputs/bbox_checks/{open_images_sample,uec_food_sample}.png`.

In [ ]:
!python -m src.data.raw_setup.visualize_raw_bboxes

In [ ]:
from IPython.display import Image, display

display(Image("outputs/bbox_checks/open_images_sample.png"))
display(Image("outputs/bbox_checks/uec_food_sample.png"))

## 3. Per-Source Conversion to YOLO Staging

Each source dataset is converted to YOLO format independently into its own
staging directory under `datasets/_staging/`. Per-source staging avoids
filename collisions between sources and lets us validate each one in
isolation before merging.

Output:
```
datasets/_staging/
  ├── uec_food/{images,labels}/
  └── open_images/{images,labels}/
reports/skipped_images/
  ├── uec_food_256.csv
  └── open_images.csv
```

### 3.1 Convert UEC FOOD-256 to YOLO

Walks the per-category folders, parses each `bb_info.txt`, converts VOC-style
bboxes to YOLO normalized xywh, and writes the staging copy. Every UEC bbox
becomes class id 0 (food).

In [ ]:
!python -m src.data.conversion.convert_uec_food_to_yolo

### 3.2 Convert Open Images to YOLO

Reads the COCO export, maps source category names to YOLO class IDs via
`configs/label_mapping.yaml`, converts COCO `[x, y, w, h]` to YOLO normalized
`[cx, cy, w, h]`, and writes the staging copy. Categories not present in
`label_mapping.yaml` are dropped.

In [ ]:
!python -m src.data.conversion.convert_open_images_to_yolo

### 3.3 Sanity Checks on Staging

Two complementary checks once both stagings exist:

1. **Visual mapping check**: render random samples per class with their
   converted YOLO boxes, so you can eyeball that `cup` boxes really wrap cups,
   `knife` wraps knives, etc. Output goes under
   `outputs/staging_bbox_checks/seed_<NN>/`.
2. **Class distribution analysis**: aggregate counts per class, share of total,
   imbalance ratio, and bbox size stats. Result is printed to stdout and
   persisted as `reports/class_distribution_<timestamp>.json` (gitignored).

In [ ]:
!python -m src.data.validation.visualize_yolo_mapping --seed 7 --samples-per-class 10

In [ ]:
!python -m src.data.validation.analyze_class_distribution

## 4. Merge and Split into Final YOLO Dataset

Two scripts run sequentially:

1. **`prepare_dataset.py`** merges both staging dirs into a single flat layout
   `datasets/table_assistant_yolo/{images,labels}/`. Each pair is renamed to
   `<NNNNNNNN>_<class_a>_<class_b>...jpg` (sequence + sorted class names) so
   the filename encodes content. Provenance lives in
   `reports/dataset_manifest.csv`. After the merge the staging dirs are
   deleted.
2. **`split_dataset.py`** produces a stratified 60/15/25 split. Stratification
   uses the rarest class present in each image, where rarity is computed from
   image counts in the dataset (not from class id). Output is three text
   files under `reports/dataset_splits/{train,val,test}.txt` containing
   absolute image paths, plus an updated
   `## Class Distribution` section in `reports/dataset_notes.md`.

YOLO consumes the splits via `configs/data.yaml`, which already points at the
three text files. No images are duplicated on disk.

### 4.1 Merge Per-Source Staging into the Final Layout

In [ ]:
!python -m src.data.preparation.prepare_dataset

### 4.2 Stratified Train/Val/Test Split

In [ ]:
!python -m src.data.preparation.split_dataset

## 5. Next Steps

Once this notebook finishes successfully:

1. **Track the dataset with DVC** from your local shell (manual, gated step):
   ```bash
   dvc add datasets/table_assistant_yolo
   git add datasets/table_assistant_yolo.dvc datasets/.gitignore
   git commit -m "track table_assistant_yolo with DVC"
   dvc push
   git push
   ```
2. **Train the model** in `02_training_colab.ipynb`. That notebook assumes the
   dataset already exists on the DVC remote and pulls it before training.